# Detecção binária no UNSW-NB15 com CNN

O modelo preserva a divisão oficial, remove duplicatas exatas e é avaliado uma única vez sobre registros de teste não vistos no treino. O alvo é `label`: `0` representa tráfego normal e `1` representa ataque.

In [ ]:
%pip install matplotlib numpy pandas seaborn ipython torch scikit-learn pyarrow

In [ ]:
from pathlib import Path
from copy import deepcopy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")

## 1. Carregamento das divisões oficiais

In [ ]:
DIRETORIO_DADOS = Path(r"dados_unsw_nb15")
CAMINHO_TREINO = DIRETORIO_DADOS / "UNSW_NB15_training-set.parquet"
CAMINHO_TESTE = DIRETORIO_DADOS / "UNSW_NB15_testing-set.parquet"

for caminho in (CAMINHO_TREINO, CAMINHO_TESTE):
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}. "
            "Execute primeiro: python 04_download_unsw_nb15.py"
        )

treino = pd.read_parquet(CAMINHO_TREINO)
teste = pd.read_parquet(CAMINHO_TESTE)

print("Antes da remoção de duplicatas:")
print("Treino:", treino.shape)
print("Teste:", teste.shape)

treino = treino.drop_duplicates().reset_index(drop=True)
teste = teste.drop_duplicates().reset_index(drop=True)

# Evita avaliar cópias exatas de registros que o modelo já viu no treino.
hashes_treino = set(pd.util.hash_pandas_object(treino, index=False))
hashes_teste = pd.util.hash_pandas_object(teste, index=False)
quantidade_sobreposta = int(hashes_teste.isin(hashes_treino).sum())
teste = teste.loc[~hashes_teste.isin(hashes_treino)].reset_index(drop=True)

print("Depois da remoção de duplicatas:")
print("Treino:", treino.shape)
print("Teste:", teste.shape)
print("Cópias do treino removidas do teste:", quantidade_sobreposta)

## 2. Separação dos atributos e do alvo

`attack_cat` é removida porque informa diretamente a categoria do ataque e causaria vazamento do alvo binário.

In [ ]:
COLUNAS_NAO_PREDITORAS = ["label", "attack_cat"]

for nome, dados in {"treino": treino, "teste": teste}.items():
    ausentes = set(COLUNAS_NAO_PREDITORAS).difference(dados.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes em {nome}: {sorted(ausentes)}")

X_treino = treino.drop(columns=COLUNAS_NAO_PREDITORAS).copy()
X_teste = teste.drop(columns=COLUNAS_NAO_PREDITORAS).copy()
y_treino = treino["label"].astype("int8")
y_teste = teste["label"].astype("int8")

if list(X_treino.columns) != list(X_teste.columns):
    raise ValueError("Os atributos de treino e teste são diferentes.")
if not set(y_treino.unique()).union(y_teste.unique()).issubset({0, 1}):
    raise ValueError("O alvo contém valores diferentes de 0 e 1.")

colunas_categoricas = X_treino.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
colunas_numericas = [
    coluna for coluna in X_treino.columns
    if coluna not in colunas_categoricas
]

X_treino[colunas_numericas] = X_treino[colunas_numericas].replace(
    [np.inf, -np.inf], np.nan
)
X_teste[colunas_numericas] = X_teste[colunas_numericas].replace(
    [np.inf, -np.inf], np.nan
)

print(f"Atributos numéricos: {len(colunas_numericas)}")
print(f"Atributos categóricos: {colunas_categoricas}")
print("Distribuição do alvo no treino:")
display(
    y_treino.value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Ataque"})
    .to_frame("quantidade")
)

## 3. Validação dentro do treino

O treino limpo é dividido em 85% para aprender os pesos e 15% para selecionar a melhor época pela perda de validação. O teste oficial permanece separado. Nesta versão, o modelo final usa os melhores pesos desse treinamento, sem retreino no conjunto completo.

In [ ]:
X_treino_cnn, X_validacao, y_treino_cnn, y_validacao = train_test_split(
    X_treino,
    y_treino,
    test_size=0.15,
    random_state=42,
    stratify=y_treino,
)

print("Treino da CNN:", X_treino_cnn.shape)
print("Validação:", X_validacao.shape)
print("Teste final:", X_teste.shape)

distribuicao = pd.DataFrame({
    "Treino CNN (%)": y_treino_cnn.value_counts(normalize=True) * 100,
    "Validação (%)": y_validacao.value_counts(normalize=True) * 100,
}).rename(index={0: "Normal", 1: "Ataque"})

display(distribuicao.round(2))

## 4. Pré-processamento para a CNN

Os atributos são os mesmos do LightGBM. Os numéricos recebem imputação pela mediana e StandardScaler; os categóricos recebem imputação e one-hot denso. O ajuste ocorre somente no treino da CNN. Validação e teste recebem apenas transform, e as matrizes são convertidas para float32.

In [ ]:
pipeline_numerico = Pipeline(steps=[
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler()),
])

pipeline_categorico = Pipeline(steps=[
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32,)),
])

preprocessamento = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, colunas_numericas),
        ("categoricas", pipeline_categorico, colunas_categoricas),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
    
)

X_treino_cnn_transformado = preprocessamento.fit_transform(
    X_treino_cnn
).astype(np.float32)

X_validacao_transformado = preprocessamento.transform(
    X_validacao
).astype(np.float32)

X_teste_transformado = preprocessamento.transform(
    X_teste
).astype(np.float32)

X_treino_tensor = torch.from_numpy(X_treino_cnn_transformado)
X_validacao_tensor = torch.from_numpy(X_validacao_transformado)
X_teste_tensor = torch.from_numpy(X_teste_transformado)


In [ ]:
def compute_metrics(y_true, y_pred, y_proba=None, model_name="CNN"):
    metricas = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    if y_proba is not None:
        metricas["ROC-AUC"] = roc_auc_score(y_true, y_proba)
    return metricas


def avaliar_em_lotes(modelo, loader, dispositivo, criterio=None):
    modelo.eval()
    probabilidades = []
    perda_total = 0.0
    quantidade = 0
    with torch.no_grad():
        for lote in loader:
            xb = lote[0].to(dispositivo)
            logits = modelo(xb).squeeze(-1)
            probabilidades.append(torch.sigmoid(logits).cpu().numpy())
            if criterio is not None:
                yb = lote[1].to(dispositivo)
                perda_total += criterio(logits, yb).item() * len(xb)
            quantidade += len(xb)
    perda = perda_total / quantidade if criterio is not None else None
    return np.concatenate(probabilidades), perda

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, in_features, out_dim=1):
        super().__init__()
        self.conv = nn.Sequential(
            # Block 1
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            # Block 2
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.2),
            # Block 3
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), 
        )
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, out_dim)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)

## 5. Configuração e treinamento

A melhor época é escolhida pela menor perda de validação. O early stopping encerra após 10 épocas sem melhora. O teste não participa dessa seleção.

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_EPOCHS = 50
BATCH_SIZE = 1024
PATIENCE = 10

y_treino_tensor = torch.from_numpy(y_treino_cnn.to_numpy(dtype=np.float32))
y_validacao_tensor = torch.from_numpy(y_validacao.to_numpy(dtype=np.float32))
gerador = torch.Generator().manual_seed(RANDOM_SEED)
loader_treino = DataLoader(
    TensorDataset(X_treino_tensor, y_treino_tensor),
    batch_size=BATCH_SIZE, shuffle=True, generator=gerador,
)
loader_validacao = DataLoader(
    TensorDataset(X_validacao_tensor, y_validacao_tensor),
    batch_size=BATCH_SIZE, shuffle=False,
)
loader_teste = DataLoader(
    TensorDataset(X_teste_tensor), batch_size=BATCH_SIZE, shuffle=False,
)

cnn = CNN1D(X_treino_cnn_transformado.shape[1], out_dim=1).to(DEVICE)
optimizer = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
criterion = nn.BCEWithLogitsLoss()
hist_cnn = {"train_loss": [], "val_loss": [], "val_acc": []}
melhor_perda = float("inf")
melhores_pesos = None
melhor_epoca = 0
epocas_sem_melhora = 0
print(f"Dispositivo: {DEVICE}")

In [ ]:
for epoch in range(1, N_EPOCHS + 1):
    cnn.train()
    perda_total = 0.0
    for xb, yb in loader_treino:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(cnn(xb).squeeze(-1), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
        optimizer.step()
        perda_total += loss.item() * len(xb)
    scheduler.step()

    probabilidades_val, vloss = avaliar_em_lotes(
        cnn, loader_validacao, DEVICE, criterion
    )
    if not np.isfinite(vloss):
        raise ValueError("Perda de validação não finita; verifique os dados e o treinamento.")
    vacc = accuracy_score(y_validacao, probabilidades_val >= 0.5)
    train_loss = perda_total / len(loader_treino.dataset)
    hist_cnn["train_loss"].append(train_loss)
    hist_cnn["val_loss"].append(vloss)
    hist_cnn["val_acc"].append(vacc)

    if vloss < melhor_perda:
        melhor_perda = vloss
        melhor_epoca = epoch
        melhores_pesos = deepcopy(cnn.state_dict())
        epocas_sem_melhora = 0
    else:
        epocas_sem_melhora += 1

    print(f"Época {epoch:02d}/{N_EPOCHS} | treino={train_loss:.4f} "
          f"| validação={vloss:.4f} | acurácia validação={vacc:.4f}")
    if epocas_sem_melhora >= PATIENCE:
        print("Early stopping: validação sem melhora.")
        break

if melhores_pesos is None:
    raise RuntimeError("Nenhum checkpoint válido foi produzido.")
cnn.load_state_dict(melhores_pesos)
print(f"Pesos restaurados da época {melhor_epoca}; perda={melhor_perda:.4f}")

## 6. Avaliação final no teste oficial tratado

Execute após finalizar a seleção pela validação. O limiar é fixado em 0,5. Não ajuste configurações com base nos resultados do teste.

In [ ]:
y_proba_cnn, _ = avaliar_em_lotes(cnn, loader_teste, DEVICE)
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
m = compute_metrics(y_teste, y_pred_cnn, y_proba_cnn, "1D CNN")
results = {"1D CNN": m}
models = {"CNN": cnn}
histories = {"CNN": hist_cnn}
final_preds = {"1D CNN": (y_pred_cnn, y_proba_cnn)}

display(pd.Series({k: v for k, v in m.items() if k != "Model"}, name="valor")
        .to_frame().style.format("{:.4f}"))
print(classification_report(
    y_teste, y_pred_cnn, labels=[0, 1],
    target_names=["Normal", "Ataque"], digits=4, zero_division=0,
))
matriz = confusion_matrix(y_teste, y_pred_cnn, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Ataque"],
            yticklabels=["Normal", "Ataque"], ax=ax)
ax.set_title("Matriz de confusão — CNN — teste oficial UNSW-NB15")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")
plt.tight_layout()
plt.show()